# NYC Mobility Analysis - Full Modeling Pipeline (coding)

This notebook contains the complete modeling workflow for:
- Data loading and preparation
- Feature engineering at zone-hour level
- Advanced tree-based models (Boosting, Bagging, Stacking)
- Performance comparison and visual diagnostics
- Hypothesis testing (zone-level, airport, borough)


In [ ]:
from __future__ import annotations
import json
import re
import zipfile
from dataclasses import dataclass
from pathlib import Path
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor, StackingRegressor, HistGradientBoostingRegressor, ExtraTreesRegressor
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor


## 1. Paths and Configuration


In [ ]:
PROJECT_ROOT = Path('..').resolve()
DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'yellow_taxi_24months_complete.xlsx'
ZONE_LOOKUP_PATH = PROJECT_ROOT / 'data' / 'external' / 'taxi_zone_lookup.csv'
RESULTS_DIR = PROJECT_ROOT / 'reports' / 'results'
FIG_DIR = PROJECT_ROOT / 'reports' / 'figures'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
MIN_ZONE_TRAIN_ROWS = 120
MAX_ZONES = 80
REL_GAIN_THRESHOLD = 0.05
AIRPORT_ZONE_IDS = {132, 138}

print('DATA_PATH:', DATA_PATH)
print('ZONE_LOOKUP_PATH exists:', ZONE_LOOKUP_PATH.exists())


## 2. Utility Functions


In [ ]:
NS = {'m':'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}

@dataclass
class SplitData:
    train: pd.DataFrame
    valid: pd.DataFrame
    test: pd.DataFrame


def excel_col_idx(cell_ref: str) -> int:
    letters = ''.join(ch for ch in cell_ref if ch.isalpha())
    n = 0
    for c in letters:
        n = n * 26 + (ord(c) - 64)
    return n - 1


def parse_xlsx_sheet(xlsx_path: Path, sheet_name: str = 'Sample_Data') -> pd.DataFrame:
    with zipfile.ZipFile(xlsx_path) as zf:
        shared_strings = []
        if 'xl/sharedStrings.xml' in zf.namelist():
            root = ET.fromstring(zf.read('xl/sharedStrings.xml'))
            for si in root.findall('m:si', NS):
                shared_strings.append(''.join((t.text or '') for t in si.findall('.//m:t', NS)))

        wb = ET.fromstring(zf.read('xl/workbook.xml'))
        rels = ET.fromstring(zf.read('xl/_rels/workbook.xml.rels'))
        rid_to_target = {rel.attrib['Id']: rel.attrib['Target'] for rel in rels}

        target = None
        for sh in wb.findall('m:sheets/m:sheet', NS):
            if sh.attrib['name'] == sheet_name:
                rid = sh.attrib['{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id']
                target = rid_to_target[rid].lstrip('/')
                if not target.startswith('xl/'):
                    target = 'xl/' + target
                break
        if target is None:
            raise ValueError(f'Sheet {sheet_name} not found')

        ws = ET.fromstring(zf.read(target))
        rows = ws.findall('m:sheetData/m:row', NS)

        parsed_rows = []
        max_col = 0
        for row in rows:
            row_vals = {}
            for c in row.findall('m:c', NS):
                ref = c.attrib.get('r', '')
                col = excel_col_idx(ref) if ref else len(row_vals)
                ctype = c.attrib.get('t')
                v = c.find('m:v', NS)
                val = v.text if v is not None else ''
                if ctype == 's' and val != '':
                    val = shared_strings[int(val)]
                elif ctype == 'inlineStr':
                    t = c.find('m:is/m:t', NS)
                    val = t.text if t is not None else ''
                row_vals[col] = val
                max_col = max(max_col, col)
            parsed_rows.append(row_vals)

    records = [[row.get(i, '') for i in range(max_col + 1)] for row in parsed_rows]
    header = records[0]
    body = records[1:]
    df = pd.DataFrame(body, columns=header)
    df.columns = [str(c).strip() if str(c).strip() else f'col_{i}' for i, c in enumerate(df.columns)]

    dedup = {}
    for c in df.columns:
        key = c.lower()
        if key not in dedup:
            dedup[key] = c
    return df[[dedup[k] for k in dedup]]


def excel_serial_to_datetime(series: pd.Series) -> pd.Series:
    return pd.to_datetime('1899-12-30') + pd.to_timedelta(series, unit='D')


def metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    mse = mean_squared_error(y_true, y_pred)
    return {
        'mae': float(mean_absolute_error(y_true, y_pred)),
        'rmse': float(np.sqrt(mse)),
        'r2': float(r2_score(y_true, y_pred)),
        'residual_variance': float(np.var(y_true - y_pred)),
    }


## 3. Data Preparation and Feature Engineering


In [ ]:
def prepare_zone_hour_data(raw: pd.DataFrame, fill_full_grid: bool = False) -> pd.DataFrame:
    required = {'tpep_pickup_datetime', 'PULocationID'}
    missing = sorted(required - set(raw.columns))
    if missing:
        raise ValueError(f'Missing required columns: {missing}')

    df = raw.copy()
    df['tpep_pickup_datetime'] = pd.to_numeric(df['tpep_pickup_datetime'], errors='coerce')
    df['PULocationID'] = pd.to_numeric(df['PULocationID'], errors='coerce')
    df = df.dropna(subset=['tpep_pickup_datetime', 'PULocationID'])

    df['PULocationID'] = df['PULocationID'].astype(int)
    df['pickup_dt'] = excel_serial_to_datetime(df['tpep_pickup_datetime'])
    df['pickup_hour'] = df['pickup_dt'].dt.floor('h')

    demand = (
        df.groupby(['pickup_hour', 'PULocationID'], as_index=False)
        .size()
        .rename(columns={'size':'demand'})
    )

    if fill_full_grid:
        all_hours = pd.date_range(demand['pickup_hour'].min(), demand['pickup_hour'].max(), freq='h')
        all_zones = np.sort(demand['PULocationID'].unique())
        grid = pd.MultiIndex.from_product([all_hours, all_zones], names=['pickup_hour', 'PULocationID'])
        demand = demand.set_index(['pickup_hour', 'PULocationID']).reindex(grid, fill_value=0).reset_index()

    demand['hour'] = demand['pickup_hour'].dt.hour
    demand['dow'] = demand['pickup_hour'].dt.dayofweek
    demand['month'] = demand['pickup_hour'].dt.month
    demand['is_weekend'] = (demand['dow'] >= 5).astype(int)
    demand['hour_sin'] = np.sin(2 * np.pi * demand['hour'] / 24)
    demand['hour_cos'] = np.cos(2 * np.pi * demand['hour'] / 24)

    demand = demand.sort_values(['PULocationID', 'pickup_hour']).reset_index(drop=True)
    grp = demand.groupby('PULocationID')

    demand['lag_1'] = grp['demand'].shift(1)
    demand['lag_2'] = grp['demand'].shift(2)
    demand['lag_24'] = grp['demand'].shift(24)
    if fill_full_grid:
        demand['lag_168'] = grp['demand'].shift(168)
        demand['roll_mean_168'] = grp['demand'].shift(1).rolling(168, min_periods=1).mean().reset_index(level=0, drop=True)
    else:
        demand['lag_168'] = demand['lag_24']
        demand['roll_mean_168'] = grp['demand'].shift(1).rolling(24, min_periods=1).mean().reset_index(level=0, drop=True)

    demand['roll_mean_24'] = grp['demand'].shift(1).rolling(24, min_periods=1).mean().reset_index(level=0, drop=True)

    lag_cols = ['lag_1','lag_2','lag_24','lag_168','roll_mean_24','roll_mean_168']
    demand[lag_cols] = demand[lag_cols].fillna(0.0)
    return demand.reset_index(drop=True)


def temporal_split(df: pd.DataFrame, time_col: str = 'pickup_hour') -> SplitData:
    uniq = np.array(sorted(df[time_col].unique()))
    n = len(uniq)
    train_end = int(n * 0.70)
    valid_end = int(n * 0.85)

    train_t = set(uniq[:train_end])
    valid_t = set(uniq[train_end:valid_end])
    test_t = set(uniq[valid_end:])

    return SplitData(
        train=df[df[time_col].isin(train_t)].copy(),
        valid=df[df[time_col].isin(valid_t)].copy(),
        test=df[df[time_col].isin(test_t)].copy(),
    )


In [ ]:
raw = parse_xlsx_sheet(DATA_PATH, sheet_name='Sample_Data')
demand = prepare_zone_hour_data(raw, fill_full_grid=False)
split = temporal_split(demand)

print('Raw shape:', raw.shape)
print('Demand shape:', demand.shape)
print('Split sizes:', len(split.train), len(split.valid), len(split.test))
print('Zones:', demand['PULocationID'].nunique(), 'Hours:', demand['pickup_hour'].nunique())

demand.head()


## 4. Model Training (Boosting, Bagging, Stacking)


In [ ]:
FEATURES = [
    'PULocationID', 'hour', 'dow', 'month', 'is_weekend',
    'hour_sin', 'hour_cos', 'lag_1', 'lag_2', 'lag_24', 'lag_168',
    'roll_mean_24', 'roll_mean_168'
]
TARGET = 'demand'

models = {
    'xgboost': XGBRegressor(
        n_estimators=120,
        max_depth=5,
        learning_rate=0.07,
        subsample=0.85,
        colsample_bytree=0.85,
        objective='reg:squarederror',
        random_state=SEED,
        n_jobs=2,
    ),
    'bagging_rf': RandomForestRegressor(
        n_estimators=160,
        max_depth=12,
        min_samples_leaf=2,
        random_state=SEED,
        n_jobs=2,
    ),
    'stacking': StackingRegressor(
        estimators=[
            ('xgb', XGBRegressor(
                n_estimators=90, max_depth=4, learning_rate=0.08,
                subsample=0.85, colsample_bytree=0.85,
                objective='reg:squarederror', random_state=SEED, n_jobs=1
            )),
            ('rf', RandomForestRegressor(n_estimators=80, random_state=SEED, n_jobs=2)),
            ('hgb', HistGradientBoostingRegressor(max_depth=6, random_state=SEED)),
            ('etr', ExtraTreesRegressor(n_estimators=80, random_state=SEED, n_jobs=2)),
        ],
        final_estimator=RidgeCV(alphas=np.logspace(-3, 3, 13)),
        passthrough=True,
        cv=3,
        n_jobs=1,
    ),
}

rows = []
preds = {}
for name, model in models.items():
    model.fit(split.train[FEATURES], split.train[TARGET])
    pred = model.predict(split.test[FEATURES])
    preds[name] = pred
    m = metrics(split.test[TARGET].to_numpy(), pred)
    m['model'] = name
    rows.append(m)

model_results = pd.DataFrame(rows).sort_values('mae').reset_index(drop=True)
model_results


## 5. Professional Performance Visualizations


In [ ]:
plt.figure(figsize=(10,6))
plot_df = model_results.copy()
plt.bar(plot_df['model'], plot_df['mae'], color=['#2F4858','#4E6E58','#6C7A89'])
plt.title('MAE Comparison - Advanced Tree-Based Models')
plt.ylabel('MAE (lower is better)')
plt.grid(axis='y', color='#CBD5E1', linewidth=0.8)
plt.show()

plt.figure(figsize=(10,6))
plt.bar(plot_df['model'], plot_df['rmse'], color=['#2F4858','#4E6E58','#6C7A89'])
plt.title('RMSE Comparison - Advanced Tree-Based Models')
plt.ylabel('RMSE (lower is better)')
plt.grid(axis='y', color='#CBD5E1', linewidth=0.8)
plt.show()

plt.figure(figsize=(10,6))
plt.bar(plot_df['model'], plot_df['r2'], color=['#2F4858','#4E6E58','#6C7A89'])
plt.title('R2 Comparison - Advanced Tree-Based Models')
plt.ylabel('R2 (higher is better)')
plt.ylim(0,1)
plt.grid(axis='y', color='#CBD5E1', linewidth=0.8)
plt.show()


## 6. Hypothesis Testing Functions


In [ ]:
def zone_specific_xgb(split: SplitData, min_train_rows: int = 120, max_zones: int = 80) -> pd.DataFrame:
    candidate_zones = (
        split.train.groupby('PULocationID').size().sort_values(ascending=False).head(max_zones).index.tolist()
    )
    for z in AIRPORT_ZONE_IDS:
        if z in set(split.train['PULocationID']) and z not in candidate_zones:
            candidate_zones.append(z)

    rows = []
    for zone in candidate_zones:
        ztr = split.train[split.train['PULocationID'] == zone]
        zte = split.test[split.test['PULocationID'] == zone]
        min_rows = 24 if zone in AIRPORT_ZONE_IDS else min_train_rows
        if len(ztr) < min_rows or len(zte) == 0:
            continue

        model = XGBRegressor(
            n_estimators=90, max_depth=4, learning_rate=0.08,
            subsample=0.85, colsample_bytree=0.85,
            objective='reg:squarederror', random_state=SEED, n_jobs=1
        )
        model.fit(ztr[FEATURES], ztr[TARGET])
        pred = model.predict(zte[FEATURES])
        m = metrics(zte[TARGET].to_numpy(), pred)
        m['PULocationID'] = int(zone)
        rows.append(m)

    return pd.DataFrame(rows)


def load_zone_lookup(path: Path) -> pd.DataFrame | None:
    if not path.exists():
        return None
    z = pd.read_csv(path)
    need = {'LocationID','Borough'}
    if not need.issubset(z.columns):
        return None
    z['LocationID'] = pd.to_numeric(z['LocationID'], errors='coerce')
    z = z.dropna(subset=['LocationID','Borough']).copy()
    z['LocationID'] = z['LocationID'].astype(int)
    return z[['LocationID','Borough']]


## 7. Run Hypothesis Tests (Zone / Airport / Borough)


In [ ]:
zone_results = zone_specific_xgb(split, min_train_rows=MIN_ZONE_TRAIN_ROWS, max_zones=MAX_ZONES)

# Zone-specific vs city-wide (XGBoost)
city_xgb = preds['xgboost']
city_test = split.test.copy()
city_test['city_xgb_pred'] = city_xgb
city_test['abs_err_city_xgb'] = (city_test[TARGET] - city_test['city_xgb_pred']).abs()
city_zone_mae = city_test.groupby('PULocationID')['abs_err_city_xgb'].mean().rename('city_xgb_zone_mae')

zone_cmp = zone_results.merge(city_zone_mae, on='PULocationID', how='left') if len(zone_results) else pd.DataFrame()
if len(zone_cmp):
    zone_cmp['mae_gain'] = zone_cmp['city_xgb_zone_mae'] - zone_cmp['mae']
    zone_support = float(zone_cmp['mae_gain'].mean()) > 0
else:
    zone_support = None

# Airport predictability
airport_support = None
airport_stats = {}
if len(zone_results):
    tmp = zone_results.copy()
    tmp['is_airport'] = tmp['PULocationID'].isin(AIRPORT_ZONE_IDS)
    g = tmp.groupby('is_airport')['residual_variance'].mean()
    av = float(g.get(True, np.nan))
    nv = float(g.get(False, np.nan))
    airport_stats = {'airport_residual_variance': av, 'non_airport_residual_variance': nv}
    if np.isfinite(av) and np.isfinite(nv):
        airport_support = av < nv

# Borough Manhattan vs Bronx
borough_support = None
borough_details = {}
zone_lookup = load_zone_lookup(ZONE_LOOKUP_PATH)
if zone_lookup is not None:
    zmap = zone_lookup.rename(columns={'LocationID':'PULocationID'})
    tr = split.train.merge(zmap, on='PULocationID', how='inner')
    te = split.test.merge(zmap, on='PULocationID', how='inner')

    borough_metrics = []
    for b, btr in tr.groupby('Borough'):
        bte = te[te['Borough'] == b]
        if len(btr) < 300 or len(bte) < 80:
            continue
        m = XGBRegressor(
            n_estimators=110, max_depth=5, learning_rate=0.08,
            subsample=0.85, colsample_bytree=0.85,
            objective='reg:squarederror', random_state=SEED, n_jobs=1
        )
        m.fit(btr[FEATURES], btr[TARGET])
        p = m.predict(bte[FEATURES])
        mm = metrics(bte[TARGET].to_numpy(), p)
        mm['Borough'] = b
        borough_metrics.append(mm)

    borough_df = pd.DataFrame(borough_metrics)
    if {'Manhattan','Bronx'}.issubset(set(borough_df['Borough'])):
        m_mae = float(borough_df.loc[borough_df['Borough']=='Manhattan','mae'].iloc[0])
        b_mae = float(borough_df.loc[borough_df['Borough']=='Bronx','mae'].iloc[0])
        rel_gain = (b_mae - m_mae) / max(b_mae, 1e-9)
        borough_support = rel_gain >= REL_GAIN_THRESHOLD
        borough_details = {'manhattan_mae': m_mae, 'bronx_mae': b_mae, 'relative_gain': rel_gain}


In [ ]:
hypothesis_summary = {
    'best_citywide_model': model_results.iloc[0]['model'],
    'zone_specific_outperform_citywide': zone_support,
    'airport_predictability_support': airport_support,
    'borough_manhattan_vs_bronx_support': borough_support,
    'airport_stats': airport_stats,
    'borough_details': borough_details,
}

print(json.dumps(hypothesis_summary, indent=2))


## 8. Save Outputs (Metrics + Hypothesis)


In [ ]:
model_results.to_csv(RESULTS_DIR / 'model_comparison.csv', index=False)
if len(zone_results):
    zone_results.to_csv(RESULTS_DIR / 'zone_specific_xgb_metrics.csv', index=False)
if 'zone_cmp' in locals() and len(zone_cmp):
    zone_cmp.to_csv(RESULTS_DIR / 'zone_vs_city_comparison.csv', index=False)

with open(RESULTS_DIR / 'hypothesis_summary.json', 'w', encoding='utf-8') as f:
    json.dump(hypothesis_summary, f, indent=2)

print('Saved outputs to', RESULTS_DIR)


## 9. Interpretation Notes

- `bagging_rf`, `xgboost`, and `stacking` are directly compared on the same temporal split.
- Zone-level and airport-level tests evaluate predictability consistency.
- Borough test requires `data/external/taxi_zone_lookup.csv`.
- Update hyperparameters or split strategy for final production experiments.
